In [ ]:
import os
import random

import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback
import csv

from loss import *
from SoftQuantizeLayer import SoftQuantizeLayer
from AnnealingScheduler import AnnealingScheduler
from DAE_DataGenerator import OptimizedDataGenerator
from dataset_utils import add_noise, quantize_manual
from plotting import *
from models import *

pi = 3.14159265359
maxval=1e9
minval=1e-9


2026-08-10 08:31:18.724985: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
NOISE_SIGMA = 0
tfrecords_base_dir = "/uscms/home/bweiss/nobackup/smart-pixels/tfrecords"
tfrecords_dir_val = os.path.join(tfrecords_base_dir, "TFR_val",'SQDAE_3srCI_0eN_catgoricalOut_Target_0_30_845')
validation_generator = OptimizedDataGenerator(
        load_from_tfrecords_dir= tfrecords_dir_val,
        shuffle=True,
        seed=3,
        quantize=False,
    )

model=CreateSQAutoencoder_Conv2D((16,16,2), n_filters=5, pool_size=3,
                        decoder_filters = 128, initial_thresholds=[10,40,800], 
                                     OutBits = 2, conv_kernel_size = 3)
weights_path = './SQ_DAE/model-20014e96-SQDAE_Conv2D_Categorical_2bit_0eN_Target_0_30_856_V2-checkpoints/'
weights_path += '/weights.1754-t124361.5547-v118714.4141.hdf5'
model.load_weights(weights_path)
model.summary()


Loading metadata from /uscms/home/bweiss/nobackup/smart-pixels/tfrecords/TFR_val/SQDAE_3srCI_0eN_catgoricalOut_Target_0_30_845/metadata.json


2026-07-07 01:54:46.097878: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-07-07 01:54:46.137273: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-07-07 01:54:46.137675: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2026-07-07 01:54:46.138620: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropri

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 16, 16, 2)]       0         
                                                                 
 soft_quantizer_output (Soft  (None, 16, 16, 2)        4         
 QuantizeLayer)                                                  
                                                                 
 q_separable_conv2d (QSepara  (None, 14, 14, 5)        33        
 bleConv2D)                                                      
                                                                 
 q_activation (QActivation)  (None, 14, 14, 5)         0         
                                                                 
 q_conv2d (QConv2D)          (None, 14, 14, 5)         30        
                                                                 
 q_activation_1 (QActivation  (None, 14, 14, 5)        0     

In [15]:
pred_cols = [str(x)+'_pred' for x in range(16*16*2)]
true_cols = [str(x)+'_true' for x in range(16*16*2)]
def construct_df_SQAutoEncoder(model, x, y):
    '''Constructs paired prediction and truth values from DAE_DG x, y for one batch'''
    this_df = pd.DataFrame()

    pred_cols = [str(x)+'_pred' for x in range(16*16*2)]
    true_cols = [str(x)+'_true' for x in range(16*16*2)]
    batch_size = len(y)
    
    truth = np.argmax(y, axis = -1).reshape(batch_size, 16*16*2)
    # print(truth.shape)
    predictions = model.predict(x, verbose = 0)
    predictions = np.argmax(predictions, axis = -1).reshape(batch_size, 16*16*2)
    # # print(predictions.shape)
    
    this_df[pred_cols] = predictions
    this_df[true_cols] = truth
    # print(len(pred_df))
    return this_df

def clusters_from_df(df):
    l = len(df)
    pred_clusters = df[pred_cols].values.reshape(l,16,16,2)
    true_clusters = df[true_cols].values.reshape(l,16,16,2)
    return pred_clusters, true_clusters

        # else:
            # complete_truth = np.concatenate((complete_truth, y), axis=0)

In [16]:
for x, y in validation_generator:
    print('---------------------')
    df = construct_df_SQAutoEncoder(model, x, y)
    print(df.tail())
    _, preds = clusters_from_df(df)
    print(preds[0][:,:,0])

---------------------
      0_pred  1_pred  2_pred  3_pred  4_pred  5_pred  6_pred  7_pred  8_pred  \
4995       0       0       0       0       0       0       0       0       0   
4996       0       0       0       0       0       0       0       0       0   
4997       0       0       0       0       0       0       0       0       0   
4998       0       0       0       0       0       0       0       0       0   
4999       0       0       0       0       0       0       0       0       0   

      9_pred  ...  502_true  503_true  504_true  505_true  506_true  507_true  \
4995       0  ...         0         0         0         0         0         0   
4996       0  ...         0         0         0         0         0         0   
4997       0  ...         0         0         0         0         0         0   
4998       0  ...         0         0         0         0         0         0   
4999       0  ...         0         0         0         0         0         0   

      508_

KeyboardInterrupt: 

In [40]:
print(len(df))
print(df.head())
print(df.tail())

103887
   0_pred  1_pred  2_pred  3_pred  4_pred  5_pred  6_pred  7_pred  8_pred  \
0       0       0       0       0       0       0       0       0       0   
1       0       0       0       0       0       0       0       0       0   
2       0       0       0       0       0       0       0       0       0   
3       0       0       0       0       0       0       0       0       0   
4       0       0       0       0       0       0       0       0       0   

   9_pred  ...  502_true  503_true  504_true  505_true  506_true  507_true  \
0       0  ...         0         0         0         0         0         0   
1       0  ...         0         0         0         0         0         0   
2       0  ...         0         0         0         0         0         0   
3       0  ...         0         0         0         0         0         0   
4       0  ...         0         0         0         0         0         0   

   508_true  509_true  510_true  511_true  
0         0      